# Кейс 5. Метрические методы классификации

Реализация с нуля: 1NN, kNN, weighted-kNN, окно Парзена (фиксированное и переменное),
профиль компактности, отбор эталонов.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from collections import Counter
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)


## 1. Ядра

In [ ]:
def kernel_uniform(r):
    return (np.abs(r) <= 1).astype(float)

def kernel_triangular(r):
    return np.maximum(0, 1 - np.abs(r))

def kernel_epanechnikov(r):
    return np.maximum(0, 0.75 * (1 - r**2))

def kernel_gaussian(r):
    return np.exp(-0.5 * r**2)

KERNELS = {
    'Uniform':      kernel_uniform,
    'Triangular':   kernel_triangular,
    'Epanechnikov': kernel_epanechnikov,
    'Gaussian':     kernel_gaussian,
}


## 2. Метрика Минковского

In [ ]:
def minkowski(x, X, p=2, w=None):
    """Взвешенная метрика Минковского: расстояние от x до каждой строки X."""
    diff = np.abs(x - X)
    if w is not None:
        diff = diff * w
    return (diff**p).sum(axis=1) ** (1.0 / p)


## 3. Классификаторы

In [ ]:
def knn_predict(x, X_train, y_train, k, p=2, w=None):
    """kNN: предсказание для одного объекта x."""
    dists = minkowski(x, X_train, p=p, w=w)
    idx = np.argsort(dists)[:k]
    return Counter(y_train[idx]).most_common(1)[0][0]

def knn_predict_batch(X_test, X_train, y_train, k, p=2, w=None):
    """kNN: предсказание для батча объектов."""
    return np.array([knn_predict(x, X_train, y_train, k, p, w) for x in X_test])

def weighted_knn_predict(x, X_train, y_train, k, p=2):
    """Взвешенный kNN: веса соседей v_i = 1/i."""
    dists = minkowski(x, X_train, p=p)
    idx = np.argsort(dists)[:k]
    labels = y_train[idx]
    weights = 1.0 / np.arange(1, k + 1)
    score = {}
    for i, lab in enumerate(labels):
        score[lab] = score.get(lab, 0) + weights[i]
    return max(score, key=score.get)

def weighted_knn_batch(X_test, X_train, y_train, k, p=2):
    return np.array([weighted_knn_predict(x, X_train, y_train, k, p) for x in X_test])

def parzen_fixed_predict(x, X_train, y_train, h, kernel, p=2):
    """Окно Парзена с фиксированной шириной h."""
    dists = minkowski(x, X_train, p=p)
    classes = np.unique(y_train)
    scores = {c: np.sum(kernel(dists[y_train == c] / h)) for c in classes}
    return max(scores, key=scores.get)

def parzen_fixed_batch(X_test, X_train, y_train, h, kernel, p=2):
    return np.array([parzen_fixed_predict(x, X_train, y_train, h, kernel, p) for x in X_test])

def parzen_variable_predict(x, X_train, y_train, k, kernel, p=2):
    """Окно Парзена с переменной шириной h(x) = rho(x, x_{(k+1)})."""
    dists = minkowski(x, X_train, p=p)
    h = np.sort(dists)[k]
    if h < 1e-12:
        h = 1e-9
    classes = np.unique(y_train)
    scores = {c: np.sum(kernel(dists[y_train == c] / h)) for c in classes}
    return max(scores, key=scores.get)

def parzen_variable_batch(X_test, X_train, y_train, k, kernel, p=2):
    return np.array([parzen_variable_predict(x, X_train, y_train, k, kernel, p) for x in X_test])


## 4. Скользящий контроль (LOO)

In [ ]:
def loo_knn(X, y, k_max=30, p=2):
    """LOO-ошибка kNN для k = 1..k_max."""
    n = len(y)
    errors = []
    for k in range(1, k_max + 1):
        err = sum(
            knn_predict(X[i], np.delete(X, i, 0), np.delete(y, i), k, p) != y[i]
            for i in range(n)
        )
        errors.append(err / n)
    return np.array(errors)

def loo_parzen_fixed(X, y, h_list, kernel, p=2):
    """LOO-ошибка метода Парзена для каждого h из h_list."""
    n = len(y)
    errors = []
    for h in h_list:
        err = sum(
            parzen_fixed_predict(X[i], np.delete(X, i, 0), np.delete(y, i), h, kernel, p) != y[i]
            for i in range(n)
        )
        errors.append(err / n)
    return np.array(errors)

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)


## 5. Профиль компактности

In [ ]:
def compactness_profile(X, y, m_max=50):
    """
    Pi(m) = доля объектов, у которых m-й ближайший сосед имеет другой класс.
    Pi(1) совпадает с LOO-ошибкой 1NN.
    """
    n = len(y)
    profile = []
    for m in range(1, min(m_max, n)):
        err = sum(
            y[np.argsort(minkowski(X[i], X))[m]] != y[i]
            for i in range(n)
        )
        profile.append(err / n)
    return np.array(profile)


## 6. Отбор эталонов

In [ ]:
def condensed_nn_add(X, y):
    """
    Жадное последовательное добавление эталонов.
    Объект добавляется, если текущий 1NN по эталонам классифицирует его неверно.
    """
    selected = []
    for i in range(len(X)):
        if not selected:
            selected.append(i)
            continue
        pred = knn_predict(X[i], X[selected], y[selected], k=1)
        if pred != y[i]:
            selected.append(i)
    return np.array(selected)

def condensed_nn_remove(X, y):
    """
    Жадное последовательное удаление эталонов.
    Объект удаляется, если без него качество на обучающей выборке не ухудшается.
    """
    selected = list(range(len(X)))
    changed = True
    while changed:
        changed = False
        for i in list(selected):
            remaining = [j for j in selected if j != i]
            if not remaining:
                continue
            # Проверяем все объекты
            all_correct = all(
                knn_predict(X[j], X[remaining], y[remaining], k=1) == y[j]
                for j in range(len(X))
            )
            if all_correct:
                selected.remove(i)
                changed = True
                break
    return np.array(selected)


## 7. Загрузка данных

In [ ]:
# Реальные датасеты
iris   = load_iris()
wine   = load_wine()
bc     = load_breast_cancer()

X_iris, y_iris = iris.data, iris.target
X_wine, y_wine = wine.data, wine.target
X_bc,   y_bc   = bc.data,   bc.target

# Стандартизация
def scale(X_tr, X_te):
    sc = StandardScaler().fit(X_tr)
    return sc.transform(X_tr), sc.transform(X_te)

# Train/test split (70/30, stratified)
splits = {}
for name, X, y in [('iris', X_iris, y_iris), ('wine', X_wine, y_wine), ('bc', X_bc, y_bc)]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_tr_sc, X_te_sc = scale(X_tr, X_te)
    splits[name] = (X_tr_sc, X_te_sc, y_tr, y_te)

# Стандартизованные полные датасеты для LOO
sc_full = {}
for name, X, y in [('iris', X_iris, y_iris), ('wine', X_wine, y_wine), ('bc', X_bc, y_bc)]:
    sc_full[name] = (StandardScaler().fit_transform(X), y)

# Синтетический датасет
X_moon, y_moon = make_moons(n_samples=200, noise=0.2, random_state=42)
X_moon_sc = StandardScaler().fit_transform(X_moon)

print("Данные загружены.")
for name, (Xtr, Xte, ytr, yte) in splits.items():
    print(f"  {name}: train={len(ytr)}, test={len(yte)}")


## 8. LOO-ошибка vs k (Iris)

In [ ]:
X_sc, y = sc_full['iris']
k_range = np.arange(1, 31)
loo_errors = loo_knn(X_sc, y, k_max=30, p=2)
best_k = k_range[np.argmin(loo_errors)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_range, loo_errors, 'o-', color='#2c7bb6', linewidth=1.5, markersize=4)
ax.axvline(best_k, color='red', linestyle='--', linewidth=1, label=f'$k^* = {best_k}$')
ax.set_xlabel('Число соседей $k$')
ax.set_ylabel('LOO-ошибка')
ax.set_title(f'Зависимость LOO-ошибки от $k$ (Iris, $p=2$)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Оптимальное k* = {best_k}, LOO = {loo_errors[best_k-1]:.4f}")


## 9. LOO-ошибка vs h (Парзен, Iris)

In [ ]:
h_list = np.linspace(0.1, 3.0, 40)
loo_h = loo_parzen_fixed(X_sc, y, h_list, kernel_epanechnikov, p=2)
best_h = h_list[np.argmin(loo_h)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h_list, loo_h, 's-', color='#d7191c', linewidth=1.5, markersize=4)
ax.axvline(best_h, color='navy', linestyle='--', linewidth=1, label=f'$h^* = {best_h:.2f}$')
ax.set_xlabel('Ширина окна $h$')
ax.set_ylabel('LOO-ошибка')
ax.set_title('Зависимость LOO-ошибки от $h$ (Iris, ядро Епанечникова)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Оптимальное h* = {best_h:.3f}, LOO = {loo_h.min():.4f}")


## 10. LOO-ошибка vs p (метрика Минковского, Iris)

In [ ]:
p_vals = [1, 2, 3, 4, 5, 6, 8, 10]
loo_p = [loo_knn(X_sc, y, k_max=int(best_k), p=pp)[int(best_k)-1] for pp in p_vals]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p_vals, loo_p, '^-', color='#1a9641', linewidth=1.5, markersize=6)
ax.set_xlabel('Параметр $p$ метрики Минковского')
ax.set_ylabel(f'LOO-ошибка при $k={best_k}$')
ax.set_title(f'Влияние параметра $p$ на LOO-ошибку (Iris, $k={best_k}$)')
ax.set_xticks(p_vals); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 11. Границы решения на make_moons

In [ ]:
k_vals_plot = [1, 5, 15]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
xx, yy = np.meshgrid(np.linspace(-2.5, 3.0, 100), np.linspace(-2.0, 2.5, 100))
grid = np.c_[xx.ravel(), yy.ravel()]

for ax, k_plot in zip(axes, k_vals_plot):
    Z = knn_predict_batch(grid, X_moon_sc, y_moon, k=k_plot).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_moon_sc[:,0], X_moon_sc[:,1], c=y_moon,
               cmap='RdBu', edgecolors='k', linewidths=0.5, s=25)
    ax.set_title(f'$k = {k_plot}$')
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

fig.suptitle('Границы решения kNN на make_moons', y=1.02)
plt.tight_layout(); plt.show()


## 12. Профиль компактности (Iris)

In [ ]:
profile = compactness_profile(X_sc, y, m_max=51)
m_range = np.arange(1, len(profile) + 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(m_range, profile, color='#7b2d8b', linewidth=1.5)
ax.set_xlabel('$m$ (номер соседа)')
ax.set_ylabel(r'$\Pi(m)$')
ax.set_title('Профиль компактности выборки Iris')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Pi(1) = {profile[0]:.4f}  (совпадает с LOO-ошибкой 1NN)")


## 13. Сравнение методов на реальных датасетах

In [ ]:
print(f"{'Датасет':<20} {'1NN':>6} {'kNN':>6} {'wkNN':>6} {'Pfix':>6} {'Pvar':>6}  k*")
print("-" * 60)

for name, (X_tr, X_te, y_tr, y_te) in splits.items():
    X_full, y_full = sc_full[name]
    loo_e = loo_knn(X_full, y_full, k_max=20, p=2)
    bk = int(np.argmin(loo_e)) + 1

    a1   = accuracy(y_te, knn_predict_batch(X_te, X_tr, y_tr, k=1))
    aknn = accuracy(y_te, knn_predict_batch(X_te, X_tr, y_tr, k=bk))
    awk  = accuracy(y_te, weighted_knn_batch(X_te, X_tr, y_tr, k=bk))
    apf  = accuracy(y_te, parzen_fixed_batch(X_te, X_tr, y_tr, h=best_h, kernel=kernel_epanechnikov))
    apv  = accuracy(y_te, parzen_variable_batch(X_te, X_tr, y_tr, k=bk, kernel=kernel_epanechnikov))

    print(f"{name:<20} {a1:>6.3f} {aknn:>6.3f} {awk:>6.3f} {apf:>6.3f} {apv:>6.3f}  k*={bk}")


## 14. Отбор эталонов (Iris)

In [ ]:
X_tr, X_te, y_tr, y_te = splits['iris']

# Полный 1NN
t0 = time.time()
pred_full = knn_predict_batch(X_te, X_tr, y_tr, k=1)
t_full = time.time() - t0
acc_full = accuracy(y_te, pred_full)

# Жадное добавление
sel_add = condensed_nn_add(X_tr, y_tr)
t0 = time.time()
pred_add = knn_predict_batch(X_te, X_tr[sel_add], y_tr[sel_add], k=1)
t_add = time.time() - t0
acc_add = accuracy(y_te, pred_add)

print(f"Полный 1NN:          accuracy={acc_full:.3f}, эталонов={len(X_tr)}, время={t_full*1000:.1f} мс")
print(f"Жадное добавление:   accuracy={acc_add:.3f}, эталонов={len(sel_add)}, время={t_add*1000:.1f} мс")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
methods = ['Полный 1NN', 'Жадное\nдобавление']
axes[0].bar(methods, [acc_full, acc_add], color=['#2c7bb6','#d7191c'], edgecolor='k')
axes[0].set_ylim(0.85, 1.01); axes[0].set_ylabel('Accuracy')
axes[0].set_title('(а) Качество классификации')
for i,v in enumerate([acc_full, acc_add]):
    axes[0].text(i, v+0.002, f'{v:.3f}', ha='center')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(methods, [len(X_tr), len(sel_add)], color=['#2c7bb6','#d7191c'], edgecolor='k')
axes[1].set_ylabel('Число эталонов')
axes[1].set_title('(б) Размер эталонного множества')
for i,v in enumerate([len(X_tr), len(sel_add)]):
    axes[1].text(i, v+0.5, str(v), ha='center')
axes[1].grid(True, alpha=0.3, axis='y')

fig.suptitle('Сравнение полного 1NN и 1NN по эталонам (Iris)')
plt.tight_layout(); plt.show()
